# Ultimate Hybrid Model

In [1]:
!pip install protobuf==4.25.3


### Cell 1: Setup and Imports
This cell imports all the libraries we'll need for the entire project .

In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import glob
import joblib
import random
import cv2
import shutil
import warnings
import argparse
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, 
    classification_report, f1_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.imagenet_utils import preprocess_input as imagenet_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print("Libraries imported.")

print("TensorFlow Version:", tf.__version__)
print("All libraries imported successfully.")

2025-11-26 12:48:09.698365: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764161289.720820     240 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764161289.727233     240 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Libraries imported.
TensorFlow Version: 2.18.0
All libraries imported successfully.


### Cell 2: Data Splitting (Run Once)
This cell takes your raw dataset and organizes it into train, val, and test folders. Update SOURCE_DATA_PATH to where your fake and real folders are.

In [ ]:
# --- CONFIGURATION FOR SPLITTING ---
# SOURCE_DATA_PATH = "/kaggle/input/faceforencispp-extracted-frames/"  # <--- UPDATE THIS PATH
# DEST_DATA_PATH = "/kaggle/working/processed_dataset/"

def split_dataset():
    if os.path.exists(DEST_DATA_PATH):
        print("Dataset already processed. Skipping split.")
        return

    print("Starting data split...")
    classes = ['fake', 'real']
    
    for class_name in classes:
        # Handle 'Deepfakes' subfolder if it exists (common in your structure)
        class_dir = os.path.join(SOURCE_DATA_PATH, class_name)
        if class_name == 'fake' and os.path.exists(os.path.join(class_dir, 'Deepfakes')):
            class_dir = os.path.join(class_dir, 'Deepfakes')
            
        if not os.path.exists(class_dir):
            print(f"Error: {class_dir} not found.")
            continue

        # Get list of VIDEO folders
        video_folders = [f for f in os.listdir(class_dir) if os.path.isdir(os.path.join(class_dir, f))]
        random.shuffle(video_folders)
        
        # 80% Train, 10% Val, 10% Test
        # train_count = int(len(video_folders) * 0.8)
        train_count = int(len(video_folders) * 0.1)
        val_count = int(len(video_folders) * 0.1)
        
        splits = {
            'train': video_folders[:train_count],
            'val': video_folders[train_count:train_count + val_count],
            'test': video_folders[train_count + val_count:]
        }
        
        print(f"Processing {class_name}: {len(video_folders)} videos found.")

        for split_type, folders in splits.items():
            save_dir = os.path.join(DEST_DATA_PATH, split_type, class_name)
            os.makedirs(save_dir, exist_ok=True)
            
            for folder in tqdm(folders, desc=f"Copying to {split_type}/{class_name}"):
                src = os.path.join(class_dir, folder)
                dst = os.path.join(save_dir, folder)
                if not os.path.exists(dst):
                    shutil.copytree(src, dst)

    print(f"\n Data ready at: {DEST_DATA_PATH}")

# split_dataset()

## Cell 2: Global Configuration
This is the main control panel for your project. You can easily modify all important parameters here.

In [4]:
# --- Training Parameters ---
TRAIN_EPOCHS = 1  # Increased since you have more data
BATCH_SIZE = 16
PATIENCE = 5

# --- Feature Selection ---
FEATURE_SELECTION_SUBSET_SIZE = 10     #1000
FS_ITERATIONS = 5                        #50

# --- File Paths (Point to the PROCESSED data) ---
BASE_DATA_PATH = "/kaggle/input/ffpp-processed-aashiq/FFPP_processed_Aashiq/FFPP_processed_Aashiq/"
TRAIN_PATH = os.path.join(BASE_DATA_PATH, "train")
VAL_PATH = os.path.join(BASE_DATA_PATH, "val")
TEST_PATH = os.path.join(BASE_DATA_PATH, "test")

# Output Paths
MODEL_OUTPUT_PATH = "/kaggle/working/models/"
FEATURE_OUTPUT_PATH = "/kaggle/working/features/"
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)
os.makedirs(FEATURE_OUTPUT_PATH, exist_ok=True)

## Cell 3: Hardware Detection
This utility function checks for available hardware (GPU/TPU) and sets the appropriate TensorFlow distribution strategy.

In [ ]:
def get_distribution_strategy():
    """
    Detects available hardware (TPU, multi-GPU, single-GPU, CPU) and returns
    the appropriate TensorFlow distribution strategy.
    """
    try:
        tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
        strategy = tf.distribute.TPUStrategy(tpu)
        print(" Running on TPU")
    except (ValueError, tf.errors.NotFoundError):
        gpus = tf.config.list_physical_devices('GPU')
        if len(gpus) > 1:
            strategy = tf.distribute.MirroredStrategy()
            print(f" Running on {len(gpus)} GPUs")
        elif len(gpus) == 1:
            strategy = tf.distribute.get_strategy()
            print(" Running on a single GPU")
        else:
            strategy = tf.distribute.get_strategy()
            print(" Running on CPU")
            
    print(f"Number of accelerator replicas: {strategy.num_replicas_in_sync}")
    return strategy

# Run the detection function
strategy = get_distribution_strategy()

✅ Running on a single GPU
Number of accelerator replicas: 1


### Cell 4: utils.py
This script handles data loading and preprocessing. It's written to handle both 'imagenet' and 'xception' preprocessing.

In [6]:
%%writefile utils.py
import os
import glob
import numpy as np
import cv2
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.applications.imagenet_utils import preprocess_input as imagenet_preprocess
from tensorflow.keras.utils import to_categorical
from sklearn.utils import shuffle

def get_files_from_structure(data_path):
    """
    Recursively finds images in subfolders (e.g. train/fake/video_01/frame.png)
    """
    fake_paths = glob.glob(os.path.join(data_path, 'fake', '**', '*.png'), recursive=True) + \
                 glob.glob(os.path.join(data_path, 'fake', '**', '*.jpg'), recursive=True)
                 
    real_paths = glob.glob(os.path.join(data_path, 'real', '**', '*.png'), recursive=True) + \
                 glob.glob(os.path.join(data_path, 'real', '**', '*.jpg'), recursive=True)
    
    file_paths = fake_paths + real_paths
    labels = [1] * len(fake_paths) + [0] * len(real_paths)
    
    if len(file_paths) == 0:
        print(f"WARNING: No images found in {data_path}")
        return [], []
        
    file_paths, labels = shuffle(file_paths, labels, random_state=42)
    print(f"Found {len(file_paths)} images in {data_path}: {len(fake_paths)} fake, {len(real_paths)} real.")
    return file_paths, labels

def load_and_prep_image(path, target_size, preprocess_type='imagenet'):
    try:
        img = cv2.imread(path)
        if img is None: return None
        img_resized = cv2.resize(img, target_size)
        if preprocess_type == 'xception':
            return xception_preprocess(img_resized)
        else:
            return imagenet_preprocess(img_resized)
    except Exception:
        return None

def image_generator(file_paths, labels, batch_size, target_size=(299, 299), preprocess_type='imagenet'):
    num_samples = len(file_paths)
    while True:
        file_paths, labels = shuffle(file_paths, labels)
        for offset in range(0, num_samples, batch_size):
            batch_paths = file_paths[offset:offset+batch_size]
            batch_labels = labels[offset:offset+batch_size]
            batch_x, batch_y = [], []
            for i, input_path in enumerate(batch_paths):
                img = load_and_prep_image(input_path, target_size, preprocess_type)
                if img is not None:
                    batch_x.append(img)
                    batch_y.append(batch_labels[i])
            if batch_x:
                yield np.array(batch_x), to_categorical(np.array(batch_y), num_classes=2)

Overwriting utils.py


Cell 5: model_attention.py
This file defines the architecture for your Xception+Attention model

In [7]:
%%writefile model_attention.py
import tensorflow as tf

def backbone():
    mod  = tf.keras.applications.Xception(weights='imagenet')
    mod = tf.keras.Model(mod.input, mod.layers[-13].output)
    return mod
    
class ModifiedBranch(tf.keras.layers.Layer):
    def __init__(self, a_vec_size, **kwargs):
        super(ModifiedBranch, self).__init__(**kwargs)
        self.a_vec_size = a_vec_size
    def build(self, input_shape):
        self.dense_layer = tf.keras.layers.Dense(self.a_vec_size, activation='tanh')
    def call(self, input):
        af = tf.keras.backend.mean(input, axis=2) 
        hs = self.dense_layer(af)
        return hs

class MainBranch(tf.keras.layers.Layer):
    def __init__(self, a_vec_size, dim, **kwargs):
        super(MainBranch, self).__init__(**kwargs)
        self.a_vec_size = a_vec_size
        self.dim = dim
    def build(self, input_shape):
        self.reshape1 = tf.keras.layers.Reshape((-1, self.a_vec_size))
        self.relu = tf.keras.activations.relu
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.reshape2 = tf.keras.layers.Reshape((self.dim**2, self.a_vec_size))
    def call(self, input):
        e = tf.transpose(input, perm=[0, 2, 1])
        e = self.reshape1(e)
        e = self.relu(e)
        e = self.dropout(e)
        e = self.reshape2(e)
        e = tf.transpose(e, perm=[0, 2, 1])
        return e

class Attention(tf.keras.layers.Layer):
    def __init__(self, dim, a_vec_size, **kwargs):
        super(Attention, self).__init__(**kwargs)
        self.dim = dim
        self.a_vec_size = a_vec_size
    def build(self, input_shape):
        self.dense1 = tf.keras.layers.Dense(self.dim**2)
        self.reshape1 = tf.keras.layers.Reshape((1, self.dim**2))
        self.add = tf.keras.layers.Add()
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.relu = tf.keras.activations.relu
        self.reshape2 = tf.keras.layers.Reshape((-1, self.a_vec_size))
        self.dense2 = tf.keras.layers.Dense(1, use_bias=False)
        self.reshape3 = tf.keras.layers.Reshape((-1, self.dim**2))
    def call(self, input):
        eh = self.dense1(input[0])
        eh = self.reshape1(eh)
        eh = self.add([input[1], eh])
        eh = self.relu(eh)
        eh = self.dropout(eh)
        eh = tf.transpose(eh, perm=[0, 2, 1])
        eh = self.reshape2(eh)
        eh = self.dense2(eh)
        eh = self.reshape3(eh)
        eh = self.relu(eh)
        return eh

def model(a_vec_size, dim):
    back = backbone()
    backbone_feature = back.output  
    out = tf.keras.layers.Conv2D(filters = a_vec_size, kernel_size = (1,1), strides=(1,1), padding = 'valid', use_bias=True)(backbone_feature)
    out = tf.keras.layers.BatchNormalization(axis=-1)(out)
    out = tf.keras.activations.relu(out)
    out = tf.keras.layers.Dropout(0.8)(out)
    out = tf.keras.layers.Reshape((a_vec_size, dim**2))(out)
    modified = ModifiedBranch(a_vec_size)(out)
    main = MainBranch(a_vec_size, dim)(out)
    att = Attention(dim, a_vec_size, name="attention_output")([modified, main])
    fin = tf.keras.layers.Dense(2, activation='softmax')(att)
    fin = tf.keras.layers.Flatten()(fin)
    mod = tf.keras.Model(inputs=back.input, outputs=fin)
    return mod

Overwriting model_attention.py


Cell 6: train_models.py (Trains ALL 4 Models)
This script trains all four deep learning models that we will use as feature extractors.

In [8]:
%%writefile train_models.py
import os
import argparse
import glob
import tensorflow as tf
from utils import get_files_from_structure, image_generator
from model_attention import model as attention_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

IMG_DIM = (299, 299)
INPUT_SHAPE = (299, 299, 3)

# Function to count images recursively (Fix for your folder structure)
def count_files_recursive(path):
    return len(glob.glob(os.path.join(path, '**', '*.png'), recursive=True)) + \
           len(glob.glob(os.path.join(path, '**', '*.jpg'), recursive=True))

def create_fine_tuned_model(model_name, num_classes=2):
    if model_name == 'DenseNet201':
        base_model = tf.keras.applications.DenseNet201(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE, pooling='avg')
    elif model_name == 'EfficientNetB5':
        base_model = tf.keras.applications.EfficientNetB5(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE, pooling='avg')
    elif model_name == 'Xception':
        base_model = tf.keras.applications.Xception(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE, pooling='avg')
    
    inputs = base_model.input
    x = base_model.output
    x = tf.keras.layers.Dense(512, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    
    model = tf.keras.Model(inputs, outputs, name=f'{model_name}_finetuned')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def run_train_backbones(args, strategy):
    print("\n--- Training Backbone Models ---")
    
    # Dynamic Counting (Important for your new dataset size)
    train_count = count_files_recursive(args.train_path)
    val_count = count_files_recursive(args.val_path)
    print(f"Training count: {train_count}, Validation count: {val_count}")
    
    train_files, train_labels = get_files_from_structure(args.train_path)
    val_files, val_labels = get_files_from_structure(args.val_path)
    
    train_gen = image_generator(train_files, train_labels, args.batch_size, IMG_DIM, 'imagenet')
    val_gen = image_generator(val_files, val_labels, args.batch_size, IMG_DIM, 'imagenet')
    
    train_steps = train_count // args.batch_size
    val_steps = val_count // args.batch_size

    for model_name in ['DenseNet201', 'EfficientNetB5', 'Xception']:
        print(f"\n*** Training {model_name} ***")
        with strategy.scope():
            model = create_fine_tuned_model(model_name)
        
        filepath = os.path.join(args.output_path, f"best_model_{model_name.lower()}.keras")
        callbacks = [
            ModelCheckpoint(filepath, save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
            EarlyStopping(monitor='val_accuracy', patience=args.patience, restore_best_weights=True)
        ]
        model.fit(train_gen, epochs=args.epochs, steps_per_epoch=train_steps, 
                  validation_data=val_gen, validation_steps=val_steps, callbacks=callbacks)

def run_train_attention(args, strategy):
    print("\n--- Training Attention Model ---")
    train_count = count_files_recursive(args.train_path)
    val_count = count_files_recursive(args.val_path)
    
    train_files, train_labels = get_files_from_structure(args.train_path)
    val_files, val_labels = get_files_from_structure(args.val_path)

    # Note: Attention model uses 'xception' preprocessing
    train_gen = image_generator(train_files, train_labels, args.batch_size, IMG_DIM, 'xception')
    val_gen = image_generator(val_files, val_labels, args.batch_size, IMG_DIM, 'xception')
    
    train_steps = train_count // args.batch_size
    val_steps = val_count // args.batch_size
    
    with strategy.scope():
        model = attention_model(a_vec_size=1024, dim=19)
        model.compile('Adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    filepath = os.path.join(args.output_path, "best_model_attention.keras")
    callbacks = [
        ModelCheckpoint(filepath, save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=args.patience, restore_best_weights=True)
    ]
    model.fit(train_gen, epochs=args.epochs, steps_per_epoch=train_steps, 
              validation_data=val_gen, validation_steps=val_steps, callbacks=callbacks)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--train_path', type=str, required=True)
    parser.add_argument('--val_path', type=str, required=True)
    parser.add_argument('--output_path', type=str, required=True)
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--batch_size', type=int, default=32)
    parser.add_argument('--patience', type=int, default=5)
    args = parser.parse_args()
    
    os.makedirs(args.output_path, exist_ok=True)
    
    strategy = tf.distribute.get_strategy()
    gpus = tf.config.list_physical_devices('GPU')
    if len(gpus) > 1: strategy = tf.distribute.MirroredStrategy()

    run_train_backbones(args, strategy)
    run_train_attention(args, strategy)

if __name__ == "__main__":
    main()

Overwriting train_models.py


Cell 7: Run Model Training
This cell executes the script from Cell 6. It will read the parameters from Cell 2 and train all four models.

In [9]:
!python train_models.py \
    --train_path {TRAIN_PATH} \
    --val_path {VAL_PATH} \
    --output_path {MODEL_OUTPUT_PATH} \
    --epochs {TRAIN_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --patience {PATIENCE}

2025-11-26 12:48:15.789601: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764161295.810111     285 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764161295.816088     285 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered

--- Training Backbone Models ---
Training count: 51005, Validation count: 6328
Found 51005 images in /kaggle/input/ffpp-processed-aashiq/FFPP_processed_Aashiq/FFPP_processed_Aashiq/train: 25452 fake, 25553 real.
Found 6328 images in /kaggle/input/ffpp-processed-aashiq/FFPP_processed_Aashiq/FFPP_processed_Aashiq/val: 3164 fake, 3164 real.

*** Training DenseNet201 ***
I0000 00:00:1764161321.502492     285 gpu_device.cc:2022] Created 

Cell 8: extract_features.py (Ultimate Hybrid)
This script loads all four trained models and extracts their features, creating the 4713-dimension "super-vector."

In [10]:
%%writefile extract_features.py
import os
import argparse
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from utils import get_files_from_structure, load_and_prep_image
from model_attention import ModifiedBranch, MainBranch, Attention

BATCH_SIZE = 32
IMG_DIM = (299, 299)

def load_all_extractors(model_path):
    print("Loading models...")
    # Load DenseNet
    m1 = tf.keras.models.load_model(os.path.join(model_path, "best_model_densenet201.keras"))
    ext_A = tf.keras.Model(inputs=m1.input, outputs=m1.layers[-4].output)
    
    # Load EfficientNet
    m2 = tf.keras.models.load_model(os.path.join(model_path, "best_model_efficientnetb5.keras"))
    ext_B = tf.keras.Model(inputs=m2.input, outputs=m2.layers[-4].output)

    # Load Xception
    m3 = tf.keras.models.load_model(os.path.join(model_path, "best_model_xception.keras"))
    ext_C = tf.keras.Model(inputs=m3.input, outputs=m3.layers[-4].output)

    # Load Attention
    custom = {"ModifiedBranch": ModifiedBranch, "MainBranch": MainBranch, "Attention": Attention}
    m4 = tf.keras.models.load_model(os.path.join(model_path, "best_model_attention.keras"), custom_objects=custom)
    attn_out = m4.get_layer('attention_output').output
    ext_D = tf.keras.Model(inputs=m4.input, outputs=tf.keras.layers.Flatten()(attn_out))

    return ext_A, ext_B, ext_C, ext_D

def extract(paths, labels, extractors):
    ext_A, ext_B, ext_C, ext_D = extractors
    all_feats, all_lbls = [], []
    
    num_batches = int(np.ceil(len(paths) / BATCH_SIZE))
    for i in tqdm(range(num_batches), desc="Extracting"):
        batch_paths = paths[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        batch_labels = labels[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        
        x_inet, x_xcept, valid_lbls = [], [], []
        for j, p in enumerate(batch_paths):
            img_i = load_and_prep_image(p, IMG_DIM, 'imagenet')
            img_x = load_and_prep_image(p, IMG_DIM, 'xception')
            if img_i is not None and img_x is not None:
                x_inet.append(img_i)
                x_xcept.append(img_x)
                valid_lbls.append(batch_labels[j])
        
        if not x_inet: continue
        
        fA = ext_A.predict(np.array(x_inet), verbose=0)
        fB = ext_B.predict(np.array(x_inet), verbose=0)
        fC = ext_C.predict(np.array(x_inet), verbose=0)
        fD = ext_D.predict(np.array(x_xcept), verbose=0)
        
        batch_feats = np.concatenate([fA, fB, fC, fD], axis=1)
        all_feats.append(batch_feats)
        all_lbls.append(np.array(valid_lbls))
        
    return np.concatenate(all_feats), np.concatenate(all_lbls)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--data_path', type=str, required=True)
    parser.add_argument('--model_path', type=str, required=True)
    parser.add_argument('--output_path', type=str, required=True)
    args = parser.parse_args()
    
    extractors = load_all_extractors(args.model_path)
    
    for split in ['train', 'val', 'test']:
        print(f"\nProcessing {split}...")
        p, l = get_files_from_structure(os.path.join(args.data_path, split))
        if not p: continue
        feats, lbls = extract(p, l, extractors)
        np.save(os.path.join(args.output_path, f"{split}_features.npy"), feats)
        np.save(os.path.join(args.output_path, f"{split}_labels.npy"), lbls)

if __name__ == "__main__":
    main()

Overwriting extract_features.py


### Cell 9: Run Feature Extraction
This runs the script from Cell 8, creating the ..._features.npy files (with 4713 features) in /kaggle/working/features/.

In [11]:
!python extract_features.py \
    --data_path {BASE_DATA_PATH} \
    --model_path {MODEL_OUTPUT_PATH} \
    --output_path {FEATURE_OUTPUT_PATH}

2025-11-26 14:01:42.098636: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764165702.120109     525 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764165702.126409     525 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading models...
I0000 00:00:1764165706.922689     525 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0

Processing train...
Found 51005 images in /kaggle/input/ffpp-processed-aashiq/FFPP_processed_Aashiq/FFPP_processed_Aashiq/train: 25452 fake, 25553 real.
Extracting:   0%|               

### Cell 10: train_knn.py (Fast & Optimized)
This is the optimized feature selection script. It will use the FEATURE_SELECTION_SUBSET_SIZE from Cell 2 and the "Fast mRMR" calculation .

In [12]:
%%writefile train_knn.py
import os
import argparse
import numpy as np
import joblib
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

def parse_args():
    parser = argparse.ArgumentParser(description='Train KNN classifier with feature selection')
    parser.add_argument('--feature_path', type=str, required=True)
    parser.add_argument('--model_path', type=str, required=True)
    parser.add_argument('--subset_size', type=int, default=1000)
    parser.add_argument('--max_iter', type=int, default=50) # <--- IT IS BACK
    return parser.parse_args()

# --- Feature Selection Logic ---

def relief_f_score(X, y, k=10):
    n_samples, n_features = X.shape
    feature_scores = np.zeros(n_features)
    for i in range(n_samples):
        distances = np.sum((X - X[i]) ** 2, axis=1)
        nearest_indices = np.argsort(distances)[1:k+1]
        hits = [idx for idx in nearest_indices if y[idx] == y[i]]
        misses = [idx for idx in nearest_indices if y[idx] != y[i]]
        for j in range(n_features):
            if hits:
                hit_diff = np.mean([abs(X[i, j] - X[idx, j]) for idx in hits])
                feature_scores[j] -= hit_diff / k
            if misses:
                miss_diff = np.mean([abs(X[i, j] - X[idx, j]) for idx in misses])
                feature_scores[j] += miss_diff / k
    return (feature_scores - np.min(feature_scores)) / (np.max(feature_scores) - np.min(feature_scores) + 1e-6)

def mrmr_score(X, y):
    relevance = mutual_info_classif(X, y)
    n_features = X.shape[1]
    redundancy = np.zeros(n_features)
    for i in range(n_features):
        if i == 0: continue
        compare_indices = np.random.choice(i, min(50, i), replace=False)
        redundancy[i] = np.mean([mutual_info_classif(X[:, [i, j]], y)[0] for j in compare_indices])
    mrmr_scores = relevance - redundancy
    return (mrmr_scores - np.min(mrmr_scores)) / (np.max(mrmr_scores) - np.min(mrmr_scores) + 1e-6)

def calculate_fitness(X_train, y_train, X_val, y_val, feature_indices, weight=0.9):
    if len(feature_indices) == 0: return 0, 0
    
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train[:, feature_indices], y_train)
    acc = knn.score(X_val[:, feature_indices], y_val)
    
    ratio = len(feature_indices) / X_train.shape[1]
    fitness = weight * acc + (1 - weight) * (1 - ratio)
    return fitness, acc

def run_feature_selection(X_train, y_train, X_val, y_val, subset_size, max_iterations):
    print("Starting Advanced Feature Selection...")
    
    # 1. Create Subset
    indices = np.random.choice(len(y_train), min(len(y_train), subset_size), replace=False)
    X_sub = X_train[indices]
    y_sub = y_train[indices]
    
    # 2. Calculate Initial Scores
    print("Calculating ReliefF & mRMR scores...")
    r_scores = relief_f_score(X_sub, y_sub)
    mi_scores = mutual_info_classif(X_sub, y_sub)
    mi_scores = (mi_scores - np.min(mi_scores)) / (np.max(mi_scores) - np.min(mi_scores) + 1e-6)
    m_scores = mrmr_score(X_sub, y_sub)
    
    combined = (r_scores + mi_scores + m_scores) / 3
    
    # 3. Iterative Optimization
    n_features = X_train.shape[1]
    current_feats = np.argsort(combined)[-int(0.3 * n_features):] # Start with top 30%
    best_feats = current_feats.copy()
    
    best_fit, best_acc = calculate_fitness(X_sub, y_sub, X_val, y_val, best_feats)
    print(f"Initial: {len(best_feats)} feats, Acc: {best_acc:.4f}")
    
    for i in range(max_iterations):
        # Exclude random 10%
        n_ex = max(1, int(0.1 * len(current_feats)))
        if len(current_feats) > n_ex:
            ex_idx = np.random.choice(len(current_feats), n_ex, replace=False)
            current_feats = np.delete(current_feats, ex_idx)
            
        # Include random best remaining
        remaining = np.setdiff1d(np.arange(n_features), current_feats)
        if len(remaining) > 0:
            n_in = min(max(1, int(0.1 * n_features)), len(remaining))
            # Pick best based on combined score
            local_best = np.argsort(combined[remaining])[-n_in:]
            current_feats = np.unique(np.concatenate([current_feats, remaining[local_best]]))
            
        fit, acc = calculate_fitness(X_sub, y_sub, X_val, y_val, current_feats)
        
        if fit > best_fit:
            best_fit = fit
            best_feats = current_feats.copy()
            print(f"Iter {i+1}: New Best! {len(best_feats)} feats, Acc: {acc:.4f}")
            
    return best_feats

def main():
    args = parse_args()
    
    print("Loading features...")
    X_train = np.load(os.path.join(args.feature_path, "train_features.npy"))
    y_train = np.load(os.path.join(args.feature_path, "train_labels.npy"))
    X_val = np.load(os.path.join(args.feature_path, "val_features.npy")) # Fixed naming
    y_val = np.load(os.path.join(args.feature_path, "val_labels.npy"))   # Fixed naming
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    
    # Run Selection
    selected_indices = run_feature_selection(
        X_train_s, y_train, X_val_s, y_val, 
        args.subset_size, args.max_iter
    )
    
    np.save(os.path.join(args.model_path, "selected_features.npy"), selected_indices)
    
    print("Training Final KNN on ALL data...")
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_s[:, selected_indices], y_train)
    
    joblib.dump(scaler, os.path.join(args.model_path, "scaler.joblib"))
    joblib.dump(knn, os.path.join(args.model_path, "knn_model.joblib"))
    print("Model Saved.")

if __name__ == "__main__":
    main()

Overwriting train_knn.py


## Cell 11: Run Feature Selection and KNN Training
This runs the script from Cell 10, using the parameters from Cell 2.

In [13]:
!python train_knn.py \
    --feature_path {FEATURE_OUTPUT_PATH} \
    --model_path {MODEL_OUTPUT_PATH} \
    --subset_size {FEATURE_SELECTION_SUBSET_SIZE} \
    --max_iter {FS_ITERATIONS}

Loading features...
Starting Advanced Feature Selection...
Calculating ReliefF & mRMR scores...
Initial: 1913 feats, Acc: 0.9894
Training Final KNN on ALL data...
Model Saved.


## Cell 12: score_knn.py (The Benchmark)
This is the benchmark cell. It loads the .npy files from the test set and runs them through the final KNN model to score performance.

In [19]:
%%writefile score_knn.py
import os
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, 
    classification_report, f1_score
)

# --- Constants ---
FEATURE_PATH = "/kaggle/working/features/"
MODEL_PATH = "/kaggle/working/models/"

def evaluate():
    print("--- Loading Test Features and Saved Models ---")
    
    try:
        X_test = np.load(os.path.join(FEATURE_PATH, "test_features.npy"))
        y_test = np.load(os.path.join(FEATURE_PATH, "test_labels.npy"))
        
        knn_model = joblib.load(os.path.join(MODEL_PATH, "knn_model.joblib"))
        scaler = joblib.load(os.path.join(MODEL_PATH, "scaler.joblib"))
        selected_indices = np.load(os.path.join(MODEL_PATH, "selected_features.npy"))
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return

    print(f"Test features: {X_test.shape}, Test labels: {y_test.shape}")
    print(f"Loaded KNN, Scaler, and {len(selected_indices)} selected features.")
    
    # --- FIX IS HERE: Scale FIRST, then Select ---
    print("Scaling full test set...")
    X_test_scaled_full = scaler.transform(X_test)
    
    print("Applying feature selection...")
    X_test_final = X_test_scaled_full[:, selected_indices]
    # ---------------------------------------------
    
    # 3. Make predictions
    print("\nRunning predictions...")
    y_pred = knn_model.predict(X_test_final)
    y_proba = knn_model.predict_proba(X_test_final)[:, 1]
    
    # 4. Metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    
    print("\n--- Test Set Evaluation Results ---")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"AUC Score: {auc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['real (0)', 'fake (1)']))
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['real', 'fake'], yticklabels=['real', 'fake'])
    plt.title('Confusion Matrix - Ultimate Hybrid Model (Test Set)', fontsize=16)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig("/kaggle/working/ultimate_hybrid_confusion_matrix.png")
    print("\nConfusion matrix saved to /kaggle/working/ultimate_hybrid_confusion_matrix.png")

if __name__ == "__main__":
    evaluate()

Overwriting score_knn.py


### Cell 13: Run Scoring
This will run the fast scoring script. You can run this right after Cell 11 is finished.

In [ ]:
!python score_knn.py

--- Loading Test Features and Saved Models ---
Test features: (6459, 6377), Test labels: (6459,)
Loaded KNN, Scaler, and 1913 selected features.
Scaling full test set...
Applying feature selection...

Running predictions...

--- Test Set Evaluation Results ---
Accuracy: 99.54%
AUC Score: 0.9969
F1 Score: 0.9954

Classification Report:
              precision    recall  f1-score   support

    real (0)       1.00      0.99      1.00      3232
    fake (1)       0.99      1.00      1.00      3227

    accuracy                           1.00      6459
   macro avg       1.00      1.00      1.00      6459
weighted avg       1.00      1.00      1.00      6459


Confusion Matrix:
[[3212   20]
 [  10 3217]]

Confusion matrix saved to /kaggle/working/ultimate_hybrid_confusion_matrix.png


### Cell 14: predict.py (Final Inference Script)
This is the final script you will download and use locally. It is built to load all 4 DL models, the KNN, the scaler, and feature indices to predict on raw image files.

In [21]:
%%writefile predict.py
import os
import argparse
import numpy as np
import tensorflow as tf
import joblib
from tqdm import tqdm
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, 
    classification_report, f1_score
)
from utils import load_and_prep_image
from model_attention import ModifiedBranch, MainBranch, Attention

# --- Constants ---
MODEL_PATH = "models/" 
IMG_DIM = (299, 299)
LABELS_MAP = {0: 'real', 1: 'fake'}
LABELS_MAP_INV = {'real': 0, 'fake': 1}

# --- Global Models ---
EXTRACTOR_A = None
EXTRACTOR_B = None
EXTRACTOR_C = None
EXTRACTOR_D = None
KNN_MODEL = None
SCALER = None
SELECTED_INDICES = None

def load_all_models():
    global EXTRACTOR_A, EXTRACTOR_B, EXTRACTOR_C, EXTRACTOR_D, KNN_MODEL, SCALER, SELECTED_INDICES
    print("Loading models...")
    
    # Load DL Models
    model_dense = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_densenet201.keras"))
    EXTRACTOR_A = tf.keras.Model(inputs=model_dense.input, outputs=model_dense.layers[-4].output)

    model_effnet = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_efficientnetb5.keras"))
    EXTRACTOR_B = tf.keras.Model(inputs=model_effnet.input, outputs=model_effnet.layers[-4].output)
    
    model_xception = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_xception.keras"))
    EXTRACTOR_C = tf.keras.Model(inputs=model_xception.input, outputs=model_xception.layers[-4].output)
    
    custom_objects = {"ModifiedBranch": ModifiedBranch, "MainBranch": MainBranch, "Attention": Attention}
    model_attn = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_attention.keras"), custom_objects=custom_objects)
    attn_output = model_attn.get_layer('attention_output').output
    EXTRACTOR_D = tf.keras.Model(inputs=model_attn.input, outputs=tf.keras.layers.Flatten()(attn_output))
    
    # Load ML Models
    KNN_MODEL = joblib.load(os.path.join(MODEL_PATH, "knn_model.joblib"))
    SCALER = joblib.load(os.path.join(MODEL_PATH, "scaler.joblib"))
    SELECTED_INDICES = np.load(os.path.join(MODEL_PATH, "selected_features.npy"))
    print("All models loaded.")

def get_prediction_data(image_path):
    img_inet = load_and_prep_image(image_path, IMG_DIM, 'imagenet')
    img_xcept = load_and_prep_image(image_path, IMG_DIM, 'xception')
    
    if img_inet is None or img_xcept is None: return None, 0.0, 0.0
        
    batch_x_inet = np.expand_dims(img_inet, axis=0)
    batch_x_xcept = np.expand_dims(img_xcept, axis=0)
    
    fA = EXTRACTOR_A.predict(batch_x_inet, verbose=0)
    fB = EXTRACTOR_B.predict(batch_x_inet, verbose=0)
    fC = EXTRACTOR_C.predict(batch_x_inet, verbose=0)
    fD = EXTRACTOR_D.predict(batch_x_xcept, verbose=0)
    
    # --- FIX IS HERE: Stack -> Scale -> Select ---
    feat_stacked = np.concatenate([fA, fB, fC, fD], axis=1) # Shape: (1, 6377)
    feat_scaled_full = SCALER.transform(feat_stacked)       # Scale full vector
    feat_final = feat_scaled_full[:, SELECTED_INDICES]      # Then select features
    # ---------------------------------------------
    
    probs = KNN_MODEL.predict_proba(feat_final)[0]
    prob_fake = probs[1]
    pred_idx = np.argmax(probs)
    return LABELS_MAP[pred_idx], probs[pred_idx] * 100, prob_fake

def evaluate_folder(folder_path):
    print(f"Scanning: {folder_path}")
    fake_paths = glob.glob(os.path.join(folder_path, 'fake', '**', '*.png'), recursive=True) + \
                 glob.glob(os.path.join(folder_path, 'fake', '**', '*.jpg'), recursive=True)
    real_paths = glob.glob(os.path.join(folder_path, 'real', '**', '*.png'), recursive=True) + \
                 glob.glob(os.path.join(folder_path, 'real', '**', '*.jpg'), recursive=True)
    
    all_paths = fake_paths + real_paths
    true_labels = [1]*len(fake_paths) + [0]*len(real_paths)
    
    if not all_paths:
        print("No images found.")
        return

    pred_labels, pred_probs = [], []
    confidences = []
    
    for i, path in enumerate(tqdm(all_paths, desc="Predicting")):
        lbl, conf, prob_fake = get_prediction_data(path)
        if lbl:
            pred_labels.append(LABELS_MAP_INV[lbl])
            pred_probs.append(prob_fake)
            confidences.append(conf)
            
    # Metrics
    if not pred_labels: return

    y_true = np.array(true_labels[:len(pred_labels)]) # Handle if some images failed
    y_pred = np.array(pred_labels)
    acc = accuracy_score(y_true, y_pred)
    
    print(f"\nAccuracy: {acc*100:.2f}%")
    print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.savefig("evaluation_confusion_matrix.png")
    print("Confusion matrix saved.")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--input_path', type=str, required=True)
    args = parser.parse_args()
    
    load_all_models()
    
    if os.path.isfile(args.input_path):
        lbl, conf, _ = get_prediction_data(args.input_path)
        print(f"Prediction: {lbl.upper()} ({conf:.2f}%)")
    elif os.path.isdir(args.input_path):
        evaluate_folder(args.input_path)

if __name__ == "__main__":
    main()

Overwriting predict.py


### Cell 15: Run Final Prediction
This cell will run the final inference script. You can use it to test the whole test set (slow) or a single image (fast).

In [22]:
# Run evaluation on the entire test set (from raw images, will be slow)
!python predict.py --input_path /kaggle/input/ffpp-processed-aashiq/FFPP_processed_Aashiq/FFPP_processed_Aashiq/test


# Or, test a single file:
# !python predict.py --input_path "/kaggle/input/1000-videos-split/1000_videos/test/fake/067_025_1.png"

2025-11-26 15:37:13.014460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764171433.035610   96476 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764171433.041776   96476 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading models...
I0000 00:00:1764171438.371624   96476 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
All models loaded.
Scanning: /kaggle/input/ffpp-processed-aashiq/FFPP_processed_Aashiq/FFPP_processed_Aashiq/test
Predicting:   0%|                                      | 0/6459 [00:00<?

In [23]:
# !ls /kaggle/input/celeb-df-preprocessed/'Celeb-DF Preprocessed'/test/fake | head -n 5
!python predict.py --input_path /kaggle/input/1000-videos-split/1000_videos/test/fake/067_025_1.png  # for celebdf

2025-11-26 16:35:33.771768: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764174933.792894  419522 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764174933.799315  419522 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading models...
I0000 00:00:1764174938.959035  419522 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
All models loaded.
I0000 00:00:1764174962.843885  419543 service.cc:148] XLA service 0x7a8c54004000 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices

In [24]:
!python predict.py --input_path /kaggle/input/1000-videos-split/1000_videos/test/real/068_9.png

2025-11-26 16:37:23.315789: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764175043.336319  419662 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764175043.342393  419662 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading models...
I0000 00:00:1764175048.479448  419662 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
All models loaded.
I0000 00:00:1764175072.537280  419686 service.cc:148] XLA service 0x7a28ec0016f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices

In [25]:
!python predict.py --input_path /kaggle/input/celeb-df-preprocessed/'Celeb-DF Preprocessed'/test/fake/id0_id17_0005_frame240_face6.jpg

2025-11-26 16:39:04.822731: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764175144.844783  419801 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764175144.850927  419801 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading models...
I0000 00:00:1764175149.930113  419801 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
All models loaded.
I0000 00:00:1764175173.859171  419822 service.cc:148] XLA service 0x7c5358001960 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices

In [27]:
!python predict.py --input_path /kaggle/input/celeb-df-preprocessed/'Celeb-DF Preprocessed'/test/real/00007_frame180_face16.jpg

2025-11-26 16:41:01.351572: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764175261.371033  419953 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764175261.377719  419953 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading models...
I0000 00:00:1764175266.551907  419953 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
All models loaded.
I0000 00:00:1764175290.662061  419974 service.cc:148] XLA service 0x7e95340033b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices

this model trained on ffpp and below predict for cross dataset

In [37]:
%%writefile predict1.py
import os
import argparse
import numpy as np
import tensorflow as tf
import joblib
from tqdm import tqdm
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix, 
    classification_report, f1_score
)
from utils import load_and_prep_image
from model_attention import ModifiedBranch, MainBranch, Attention

# --- Constants ---
MODEL_PATH = "/kaggle/working/models" 
IMG_DIM = (299, 299)
LABELS_MAP = {0: 'real', 1: 'fake'}
LABELS_MAP_INV = {'real': 0, 'fake': 1}

# --- Global Models ---
EXTRACTOR_A = None
EXTRACTOR_B = None
EXTRACTOR_C = None
EXTRACTOR_D = None
KNN_MODEL = None
SCALER = None
SELECTED_INDICES = None

def load_all_models():
    global EXTRACTOR_A, EXTRACTOR_B, EXTRACTOR_C, EXTRACTOR_D, KNN_MODEL, SCALER, SELECTED_INDICES
    print("Loading models...")
    
    # Load DL Models
    model_dense = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_densenet201.keras"))
    EXTRACTOR_A = tf.keras.Model(inputs=model_dense.input, outputs=model_dense.layers[-4].output)

    model_effnet = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_efficientnetb5.keras"))
    EXTRACTOR_B = tf.keras.Model(inputs=model_effnet.input, outputs=model_effnet.layers[-4].output)
    
    model_xception = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_xception.keras"))
    EXTRACTOR_C = tf.keras.Model(inputs=model_xception.input, outputs=model_xception.layers[-4].output)
    
    custom_objects = {"ModifiedBranch": ModifiedBranch, "MainBranch": MainBranch, "Attention": Attention}
    model_attn = tf.keras.models.load_model(os.path.join(MODEL_PATH, "best_model_attention.keras"), custom_objects=custom_objects)
    attn_output = model_attn.get_layer('attention_output').output
    EXTRACTOR_D = tf.keras.Model(inputs=model_attn.input, outputs=tf.keras.layers.Flatten()(attn_output))
    
    # Load ML Models
    KNN_MODEL = joblib.load(os.path.join(MODEL_PATH, "knn_model.joblib"))
    SCALER = joblib.load(os.path.join(MODEL_PATH, "scaler.joblib"))
    SELECTED_INDICES = np.load(os.path.join(MODEL_PATH, "selected_features.npy"))
    print("All models loaded.")

def get_prediction_data(image_path):
    img_inet = load_and_prep_image(image_path, IMG_DIM, 'imagenet')
    img_xcept = load_and_prep_image(image_path, IMG_DIM, 'xception')
    
    if img_inet is None or img_xcept is None: return None, 0.0, 0.0
        
    batch_x_inet = np.expand_dims(img_inet, axis=0)
    batch_x_xcept = np.expand_dims(img_xcept, axis=0)
    
    fA = EXTRACTOR_A.predict(batch_x_inet, verbose=0)
    fB = EXTRACTOR_B.predict(batch_x_inet, verbose=0)
    fC = EXTRACTOR_C.predict(batch_x_inet, verbose=0)
    fD = EXTRACTOR_D.predict(batch_x_xcept, verbose=0)
    
    # --- Feature Stacking & Selection ---
    feat_stacked = np.concatenate([fA, fB, fC, fD], axis=1)
    feat_scaled_full = SCALER.transform(feat_stacked)
    feat_final = feat_scaled_full[:, SELECTED_INDICES]
    # ------------------------------------
    
    probs = KNN_MODEL.predict_proba(feat_final)[0]
    prob_fake = probs[1]
    pred_idx = np.argmax(probs)
    return LABELS_MAP[pred_idx], probs[pred_idx] * 100, prob_fake

def evaluate_folder(folder_path):
    """
    Evaluates images in a folder. 
    Auto-detects if 'test' subdirectory exists (dataset root structure) 
    or processes the given folder directly.
    """
    
    # Check if the user provided the root '1000_videos' folder containing 'test'
    test_split_path = os.path.join(folder_path, 'test')
    
    target_path = folder_path
    if os.path.isdir(test_split_path):
        print(f"Dataset root detected. Switching target to test split: {test_split_path}")
        target_path = test_split_path
        
    print(f"Scanning directory: {target_path}")

    # Gather Real and Fake images
    # We use recursive=True to handle nested folders if they exist, 
    # but it also works for the flat file structure provided.
    fake_paths = glob.glob(os.path.join(target_path, 'fake', '**', '*.png'), recursive=True) + \
                 glob.glob(os.path.join(target_path, 'fake', '**', '*.jpg'), recursive=True)
    
    real_paths = glob.glob(os.path.join(target_path, 'real', '**', '*.png'), recursive=True) + \
                 glob.glob(os.path.join(target_path, 'real', '**', '*.jpg'), recursive=True)
    
    all_paths = fake_paths + real_paths
    # 1 for fake, 0 for real
    true_labels = [1]*len(fake_paths) + [0]*len(real_paths)
    
    if not all_paths:
        print(f"No images found in {target_path}. \nExpected structure: {target_path}/fake/image.png and {target_path}/real/image.png")
        return

    print(f"Found {len(fake_paths)} Fake and {len(real_paths)} Real images.")

    pred_labels, pred_probs = [], []
    confidences = []
    
    for i, path in enumerate(tqdm(all_paths, desc="Predicting")):
        lbl, conf, prob_fake = get_prediction_data(path)
        if lbl:
            pred_labels.append(LABELS_MAP_INV[lbl])
            pred_probs.append(prob_fake)
            confidences.append(conf)
            
    # Metrics
    if not pred_labels: return

    y_true = np.array(true_labels[:len(pred_labels)]) # Handle if some images failed to load
    y_pred = np.array(pred_labels)
    acc = accuracy_score(y_true, y_pred)
    
    print(f"\nAccuracy: {acc*100:.2f}%")
    print(classification_report(y_true, y_pred, target_names=['Real', 'Fake']))
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    save_path = os.path.join(target_path, "evaluation_confusion_matrix.png")
    plt.savefig("evaluation_confusion_matrix.png")
    print(f"Confusion matrix saved to evaluation_confusion_matrix.png")

def main():
    parser = argparse.ArgumentParser(description="Deepfake Prediction Script")
    parser.add_argument('--input_path', type=str, required=True, 
                        help="Path to single image OR dataset root folder (e.g., 1000_videos/)")
    args = parser.parse_args()
    
    load_all_models()
    
    if os.path.isfile(args.input_path):
        # Single Image Prediction
        lbl, conf, _ = get_prediction_data(args.input_path)
        print(f"Prediction: {lbl.upper()} ({conf:.2f}%)")
    elif os.path.isdir(args.input_path):
        # Folder Evaluation
        evaluate_folder(args.input_path)
    else:
        print("Invalid input path.")

if __name__ == "__main__":
    main()

Overwriting predict1.py


In [38]:
!python predict1.py --input_path /kaggle/input/celeb-df-preprocessed/'Celeb-DF Preprocessed'/test/real/00007_frame180_face16.jpg

2025-11-26 16:51:52.167575: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764175912.189407  420267 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764175912.195983  420267 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading models...
I0000 00:00:1764175917.381548  420267 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
All models loaded.
I0000 00:00:1764175941.543680  420290 service.cc:148] XLA service 0x7c51dc004c20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices

In [39]:
!python predict1.py --input_path /kaggle/input/celeb-df-preprocessed/'Celeb-DF Preprocessed'/test
!python predict1.py --input_path /kaggle/input/1000-videos-split/1000_videos/test

2025-11-26 16:53:18.697766: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764175998.718619  420406 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764175998.724944  420406 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading models...
I0000 00:00:1764176003.797337  420406 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
All models loaded.
Scanning directory: /kaggle/input/celeb-df-preprocessed/Celeb-DF Preprocessed/test
Found 4982 Fake and 1558 Real images.
Predicting:   0%|                             

In [ ]:
!python predict1.py --input_path /kaggle/input/1000-videos-split/1000_videos/test